<a href="https://colab.research.google.com/github/tannishgarg01/ACT-104-Week-2-/blob/main/ACT_104_Week_2_Improved_(Tannish_Garg).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 — Multimodel Chat Playground

**Name:** Tannish Garg

This notebook uses **OpenRouter** as a single gateway to compare multiple LLMs on the same three tasks:
1. Creative Writing
2. Analytical Reasoning
3. Instruction Following

The same prompt is sent to every selected model so that the model is the main variable in the comparison.

> **Security:** The API key is entered at runtime with `getpass` and is never stored in the notebook.


In [1]:
# Install the OpenAI-compatible client used by OpenRouter.
!pip -q install openai


In [7]:
from getpass import getpass
from openai import OpenAI
import json
import time
import re
from IPython.display import display, Markdown

# Enter your key when the cell runs. Do NOT type it into notebook source code.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Models from different providers, routed through the same OpenRouter endpoint.
OPENAI_MODEL = "openai/gpt-4o-mini"
GEMINI_MODEL = "google/gemini-2.5-flash"
CLAUDE_MODEL = "anthropic/claude-sonnet-4.6"

print(CLAUDE_MODEL)
print("API key loaded:", bool(OPENROUTER_API_KEY))
print("Models configured:", OPENAI_MODEL, GEMINI_MODEL, CLAUDE_MODEL)


Enter your OpenRouter API key: ··········
anthropic/claude-sonnet-4.6
API key loaded: True
Models configured: openai/gpt-4o-mini google/gemini-2.5-flash anthropic/claude-sonnet-4.6


## Student Functions

The assignment expects three functions. All three use the same OpenRouter API key and endpoint, but each function routes the request to a different model.

The functions return **plain text only** and do not print inside the function.


In [8]:
def _call_openrouter(prompt: str, model: str, temperature: float = 0.3, max_tokens: int = 250) -> str:
    """Send one prompt to one OpenRouter model and return its text response."""
    client = OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
    )

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content or ""


def call_openai(prompt: str) -> str:
    """Call the selected OpenAI model through OpenRouter."""
    return _call_openrouter(prompt, OPENAI_MODEL)


def call_gemini(prompt: str) -> str:
    """Call the selected Gemini model through OpenRouter."""
    return _call_openrouter(prompt, GEMINI_MODEL)


def call_claude(prompt: str) -> str:
    """Call the selected Claude model through OpenRouter."""
    return _call_openrouter(prompt, CLAUDE_MODEL)


In [9]:
print(CLAUDE_MODEL)

anthropic/claude-sonnet-4.6


In [10]:
# Quick sanity check
print("OpenAI:", call_openai("Say hi in 3 words."))
print("Gemini:", call_gemini("Say hi in 3 words."))
print("Claude:", call_claude("Say hi in 3 words."))


OpenAI: Hello there, friend!
Gemini: Hello there!
Claude: "Hi there, friend!" 👋


## 1. Creative Writing

**Prompt:** Write a 4-line poem about learning by testing code.

Evaluation focus:
- Creativity
- Tone
- Originality
- Whether the model follows the 4-line constraint


In [11]:
CREATIVE_PROMPT = "Write a 4-line poem about learning by testing code."


In [18]:
show_results("CREATIVE WRITING RESULTS", creative_results)

CREATIVE WRITING RESULTS

MODEL: OpenAI GPT-4o-mini
Response time: 0.972 seconds
OUTPUT:
In lines of code, we weave our dreams,  
With every test, a lesson gleams.  
Errors teach what books can't show,  
In trial and error, our skills will grow.

MODEL: Google Gemini 2.5 Flash
Response time: 0.733 seconds
OUTPUT:
When code resists, a bug unseen,
We test and prod, a new routine.
Each failure lights the path anew,
Until the perfect fix shines through.

MODEL: Anthropic Claude Sonnet 4.6
Response time: 2.129 seconds
OUTPUT:
Here's a 4-line poem about learning by testing code:

*You write a line and watch it break and fail,*
*Each error lights the path you need to take,*
*Through trial and run, the lessons never pale,*
*You learn the most from every bug you make.*


## 2. Analytical Reasoning

**Prompt:** Summarize the following idea in one sentence and keep it within 40 words.

Evaluation focus:
- Accuracy
- Conciseness
- Clarity
- Sentence/word-count compliance


In [19]:
ANALYTICAL_PROMPT = (
    "Summarize in one sentence (<= 40 words): "
    "'Using multiple LLM providers helps compare creativity, formatting reliability, and speed, "
    "but adds integration overhead and testing complexity.'"
)


## 3. Instruction Following

**Prompt:** Return only valid JSON with an exact schema.

Evaluation focus:
- Valid JSON
- Exact key names
- Exactly two list items
- Correct data types
- No Markdown fences or extra explanation


In [20]:
INSTRUCTION_PROMPT = (
    "Return ONLY valid JSON (no backticks). The JSON must be an object with key 'todos' "
    "which is a list of EXACTLY 2 items. Each item must be an object with keys 'task' (string) "
    "and 'done' (boolean)."
)


## Comparative Playtest

The function below sends the **same prompt** to all three models and records response time and output.

This lets us compare model behavior while keeping the task constant.


In [21]:
MODEL_FUNCTIONS = {
    "OpenAI GPT-4o-mini": call_openai,
    "Google Gemini 2.5 Flash": call_gemini,
    "Anthropic Claude Sonnet 4.6": call_claude,
}

def run_comparison(prompt: str):
    results = {}

    for model_name, function in MODEL_FUNCTIONS.items():
        start = time.perf_counter()
        try:
            output = function(prompt)
            error = None
        except Exception as exc:
            output = ""
            error = str(exc)

        elapsed = time.perf_counter() - start

        results[model_name] = {
            "output": output,
            "seconds": round(elapsed, 3),
            "error": error,
        }

    return results


In [22]:
creative_results = run_comparison(CREATIVE_PROMPT)
analytical_results = run_comparison(ANALYTICAL_PROMPT)
instruction_results = run_comparison(INSTRUCTION_PROMPT)


In [23]:
def show_results(title, results):
    print("=" * 80)
    print(title)
    print("=" * 80)

    for model, result in results.items():
        print(f"\nMODEL: {model}")
        print(f"Response time: {result['seconds']} seconds")

        if result["error"]:
            print("ERROR:", result["error"])
        else:
            print("OUTPUT:")
            print(result["output"])

show_results("CREATIVE WRITING RESULTS", creative_results)
show_results("ANALYTICAL REASONING RESULTS", analytical_results)
show_results("INSTRUCTION FOLLOWING RESULTS", instruction_results)


CREATIVE WRITING RESULTS

MODEL: OpenAI GPT-4o-mini
Response time: 0.99 seconds
OUTPUT:
In lines of code, we weave our dreams,  
With every test, the logic gleams.  
Errors teach what words can't say,  
In trial and error, we find our way.

MODEL: Google Gemini 2.5 Flash
Response time: 0.758 seconds
OUTPUT:
When code resists, a bug unseen,
We test and prod, a new routine.
Each failure lights a path to mend,
Until the perfect fix, we find.

MODEL: Anthropic Claude Sonnet 4.6
Response time: 2.058 seconds
OUTPUT:
Here's a 4-line poem about learning by testing code:

*You write a line and watch it break and bend,*
*Each error message guides you to the end,*
*Through trial and run, the lessons start to show,*
*You test, you fail, you fix — and then you know.*
ANALYTICAL REASONING RESULTS

MODEL: OpenAI GPT-4o-mini
Response time: 0.944 seconds
OUTPUT:
Utilizing multiple LLM providers enhances creativity, formatting reliability, and speed comparison, but introduces challenges in integration a

## **Comparative Analysis of Experimental Results**

### **1. Creative Writing**

All three models successfully generated a four-line poem about learning by testing code.

GPT-4o-mini produced a motivational poem that emphasized learning through mistakes and practical experience. Its rhyming structure made the poem easy to read and understand.

Gemini 2.5 Flash focused more on debugging, failure, and finding solutions. It followed the four-line requirement directly and produced the fastest response in 0.733 seconds.

Claude Sonnet 4.6 produced an expressive poem with an alternating rhyme pattern. However, it included an introductory sentence before the poem and took 2.129 seconds.

### **2. Analytical Reasoning**

All three models successfully summarized the given information in one sentence within the 40-word limit.

GPT-4o-mini highlighted creativity, formatting reliability, and integration challenges.

Gemini provided a concise summary and achieved the fastest response time of 0.574 seconds.

Claude clearly explained the relationship between the benefits of comparing multiple providers and the additional integration overhead. Its wording closely preserved the meaning of the original statement.

### **3. Instruction Following**

GPT-4o-mini and Claude successfully returned raw JSON with the required keys, exactly two items, and correct data types.

Gemini generated the correct JSON structure but enclosed it within Markdown code fences. This violated the explicit instruction to return only valid JSON without backticks.

This difference is important because automated applications may fail to parse responses containing additional formatting.

### **4. Response Time Comparison**

| Model | Creative Writing | Analytical Reasoning | Instruction Following | Average |
|---|---:|---:|---:|---:|
| GPT-4o-mini | 0.972 s | 0.964 s | 1.115 s | 1.017 s |
| Gemini 2.5 Flash | 0.733 s | 0.574 s | 0.850 s | 0.719 s |
| Claude Sonnet 4.6 | 2.129 s | 1.057 s | 1.069 s | 1.418 s |

### **5. Overall Findings**
Gemini demonstrated the fastest response times across all three experiments, while GPT-4o-mini provided a good balance between speed, creativity, and instruction following.

Claude produced expressive creative writing and accurate structured outputs but had the highest average response time.

The experiments show that model selection depends on application requirements. Speed, response quality, formatting reliability, and instruction following are important factors.

Since each task was tested only once, these results represent the observed performance in this experiment rather than a universal ranking of the models.

## Automatic Checks

These checks mirror the important formatting requirements in the starter assignment.

They do **not** decide which model is better; they only verify whether each response satisfies the explicit task constraints.


In [24]:
def creative_check(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return len(lines) == 4


def analytical_check(text):
    text = text.strip()
    words = text.split()

    # Simple sentence count for this controlled task.
    sentence_endings = len(re.findall(r"[.!?](?:\s|$)", text))

    return (
        len(words) <= 40
        and sentence_endings == 1
        and text.endswith((".", "!", "?"))
    )


def instruction_check(text):
    try:
        data = json.loads(text)
    except Exception:
        return False

    if set(data.keys()) != {"todos"}:
        return False

    todos = data["todos"]
    if not isinstance(todos, list) or len(todos) != 2:
        return False

    for item in todos:
        if set(item.keys()) != {"task", "done"}:
            return False
        if not isinstance(item["task"], str):
            return False
        if not isinstance(item["done"], bool):
            return False

    return True


def build_check_table(results, checker):
    rows = []
    for model, result in results.items():
        passed = False if result["error"] else checker(result["output"])
        rows.append({
            "Model": model,
            "Seconds": result["seconds"],
            "Format check": "PASS" if passed else "FAIL",
        })
    return rows


In [25]:
import pandas as pd

summary = pd.DataFrame(
    build_check_table(creative_results, creative_check)
    + build_check_table(analytical_results, analytical_check)
    + build_check_table(instruction_results, instruction_check)
)

summary["Task"] = (
    ["Creative"] * len(MODEL_FUNCTIONS)
    + ["Analytical"] * len(MODEL_FUNCTIONS)
    + ["Instruction"] * len(MODEL_FUNCTIONS)
)

summary = summary[["Task", "Model", "Seconds", "Format check"]]
display(summary)


,Task,Model,Seconds,Format check
0,Creative,OpenAI GPT-4o-mini,0.990,PASS
1,Creative,Google Gemini 2.5 Flash,0.758,PASS
2,Creative,Anthropic Claude Sonnet 4.6,2.058,FAIL
3,Analytical,OpenAI GPT-4o-mini,0.944,PASS
4,Analytical,Google Gemini 2.5 Flash,0.631,PASS
5,Analytical,Anthropic Claude Sonnet 4.6,1.120,PASS
6,Instruction,OpenAI GPT-4o-mini,1.112,PASS
7,Instruction,Google Gemini 2.5 Flash,0.699,FAIL
8,Instruction,Anthropic Claude Sonnet 4.6,1.166,PASS


## **Manual Comparison Notes**

| Task | Model Observations |
|---|---|
| Creative Writing | GPT-4o-mini produced a motivational and well-rhymed poem. Gemini focused on debugging and learning from failures. Claude produced an expressive poem but failed the strict formatting check because of additional text in the earlier displayed output. |
| Analytical Reasoning | All three models passed the one-sentence and word-limit checks. Gemini provided a concise response, while Claude closely preserved the original meaning. GPT-4o-mini highlighted the benefits and integration challenges. |
| Instruction Following | GPT-4o-mini and Claude passed the JSON formatting checks. Gemini failed the formatting check and its earlier displayed response contained unwanted Markdown code fences. |
| Speed | Gemini was the fastest model across all three tasks. Based on the latest summary table, its average response time was 0.696 seconds, followed by GPT-4o-mini at 1.015 seconds and Claude at 1.448 seconds. |
| Reliability | GPT-4o-mini passed all three automatic checks. Gemini and Claude each passed two out of three checks. These results demonstrate differences in formatting reliability and instruction following. |

### Overall Observation

GPT-4o-mini provided the best balance of speed and formatting reliability in this experiment. Gemini was the fastest, while Claude demonstrated expressive writing and accurate structured output. The most suitable model depends on the specific application requirements.

## **Reflection — 700–900 Words**

### **Introduction**

In this assignment, I developed a multimodel chat playground using Python and OpenRouter. The main objective was to understand how different large language models respond to the same prompts and how their performance varies depending on the task. I selected three models: OpenAI GPT-4o-mini, Google Gemini 2.5 Flash, and Anthropic Claude Sonnet 4.6.

I tested these models on three different tasks: creative writing, analytical reasoning, and instruction following. I used the same prompt for each model to make the comparison fairer. I also recorded response times and implemented automatic formatting checks to evaluate their performance.

### **Creative Writing**

For the first task, I asked each model to write a four-line poem about learning by testing code. I wanted to compare their creativity, tone, originality, and ability to follow the requested format.

GPT-4o-mini generated a motivational poem that focused on learning through mistakes and practical experience. I particularly liked the line, "Errors teach what books can't show," because it connects programming with the importance of practical learning. The poem had a clear rhyming structure and followed the four-line requirement.

Gemini 2.5 Flash produced a poem that focused more on debugging, identifying problems, and finding solutions. Its response was simple, relevant, and easy to understand. It also followed the four-line requirement correctly.

Claude Sonnet 4.6 generated an expressive poem that highlighted the importance of learning from failures. However, it included an introductory sentence before the poem. Although the poem itself contained four lines, the complete response failed the strict automatic formatting check.

This task helped me understand that creative quality and instruction following are different aspects of model performance. A creative response may still fail when an application requires a specific output format.

### **Analytical Reasoning**

For the second task, I asked the models to summarize a statement about using multiple LLM providers in one sentence within 40 words.

All three models successfully followed the sentence and word-limit requirements. GPT-4o-mini explained the benefits of comparing creativity, formatting reliability, and speed while mentioning integration and testing challenges.

Gemini provided a concise summary that communicated both the benefits and limitations without unnecessary explanation. Claude closely preserved the original meaning by explaining that multiple providers allow performance comparisons but increase integration overhead and testing complexity.

I noticed that GPT-4o-mini used the word "enhances," which slightly changes the original meaning. Using multiple providers helps compare creativity rather than necessarily improving creativity itself. Claude's wording was closer to the original statement.

This experiment showed me that analytical responses should be evaluated not only on their length and clarity but also on how accurately they preserve the meaning of the original information.

### **Instruction Following**

The third task required the models to return only valid JSON containing exactly two todo items with specific keys and data types.

GPT-4o-mini returned correctly structured JSON without additional explanations. Claude also returned valid JSON in a compact format and followed the required schema.

Gemini generated the requested JSON content, but its earlier displayed response included Markdown code fences even though the prompt explicitly prohibited them. It also failed the latest automatic formatting check.

This difference is important when developing real-world applications. If an application expects raw JSON, additional Markdown formatting can cause parsing errors. Developers may need to implement validation, output cleaning, or structured response mechanisms before using generated content.

### **Response Time and Reliability**

Based on my latest experimental results, Gemini achieved the lowest average response time of approximately 0.696 seconds. GPT-4o-mini recorded an average of 1.015 seconds, while Claude recorded approximately 1.448 seconds.

Gemini was the fastest model across all three tasks in this run. However, GPT-4o-mini demonstrated stronger formatting reliability by passing all three automatic checks. Gemini and Claude each passed two out of three checks.

These observations helped me understand that speed alone does not determine which model is most suitable. An application that requires strict formatting may prioritize reliability, while a conversational application may prioritize faster responses.

Since the experiment included only a small number of requests, these timings cannot establish permanent performance rankings. Network conditions, server load, and provider availability may influence response times.

### **Challenges and Personal Learning**

One challenge I faced was model availability. The original Claude Sonnet 4 model returned a 404 error because it was no longer available through OpenRouter. I resolved this issue by updating the model identifier to Claude Sonnet 4.6.

This experience helped me understand the importance of checking model availability and handling API errors. I also learned how a common interface makes switching between model providers easier.

Although OpenRouter simplified integration, using a single gateway is different from directly integrating separate provider APIs. Direct integrations could introduce additional authentication, configuration, and maintenance requirements.

I also learned that automatic checks are useful for evaluating formatting requirements, but human judgment remains important when assessing creativity, accuracy, and clarity.

### **Conclusion**

Overall, this assignment improved my understanding of multimodel LLM applications. GPT-4o-mini provided a good balance between response speed and formatting reliability, Gemini demonstrated faster responses, and Claude produced expressive writing and accurate structured output.

My main learning is that there is no single model that is necessarily best for every application. Model selection should depend on the task, response quality, reliability, speed, and integration requirements. In future projects, I would conduct more repeated experiments and implement stronger validation and error-handling mechanisms before selecting a model for production use.

## **AI Usage Disclosure**

I used ChatGPT (GPT-6.2) by OpenAI to assist with debugging, code improvement, comparative analysis, and drafting and organizing my reflection.

The experimental responses were generated by executing OpenAI GPT-4o-mini, Google Gemini 2.5 Flash, and Anthropic Claude Sonnet 4.6 through OpenRouter in Google Colab.

**Example input to the models:**

"Write a 4-line poem about learning by testing code."

**Actual GPT-4o-mini output:**

In lines of code, we weave our dreams,  
With every test, a lesson gleams.  
Errors teach what books can't show,  
In trial and error, our skills will grow.

The comparative analysis is based on actual model outputs, measured response times, and automatic formatting checks.

### **Additional Experiment — Original Prompt Comparison**

This experiment compares the three models using an original prompt to evaluate their response quality and instruction-following abilities.

In [26]:
ORIGINAL_PROMPT = input("Enter your original prompt: ")

original_results = run_comparison(ORIGINAL_PROMPT)

show_results("ORIGINAL PROMPT COMPARISON", original_results)

Enter your original prompt: Explain the difference between arrays and linked lists in exactly 3 bullet points using simple English and no more than 60 words.
ORIGINAL PROMPT COMPARISON

MODEL: OpenAI GPT-4o-mini
Response time: 2.176 seconds
OUTPUT:
- **Structure**: Arrays store elements in a fixed-size, contiguous block of memory, while linked lists consist of nodes that point to each other, allowing for dynamic size.  
- **Access**: Arrays allow quick access to elements using an index, whereas linked lists require traversing from the start to find an element.  
- **Memory**: Arrays use a single memory allocation, while linked lists use separate allocations for each node, which can lead to more overhead.

MODEL: Google Gemini 2.5 Flash
Response time: 1.048 seconds
OUTPUT:
*   **Arrays are like numbered boxes:** You know exactly where everything is, but adding or removing boxes in the middle is a hassle.
*   **Linked lists are like a treasure hunt:** Each item tells you where to find th

## **Additional Experiment — Comparative Analysis**

**Prompt Used:**

Explain the difference between arrays and linked lists in exactly 3 bullet points using simple English and no more than 60 words.

### **GPT-4o-mini**

GPT-4o-mini provided a technical explanation focusing on memory structure, element access, and memory overhead. It followed the three-bullet requirement but exceeded the 60-word limit. Its response time was 2.176 seconds.

### **Gemini 2.5 Flash**

Gemini used simple examples such as numbered boxes and a treasure hunt to explain the differences. These examples made the concepts easier for beginners to understand. However, it exceeded the word limit and oversimplified the ease of inserting and deleting linked list elements. It was the fastest model, with a response time of 1.048 seconds.

### **Claude Sonnet 4.6**

Claude provided a clear explanation of memory organization, element access, and insertion and deletion operations. It followed the three-bullet requirement but exceeded the word limit. Its response time was 2.242 seconds.

### **Overall Findings**

All three models followed the three-bullet requirement but failed to stay within the requested 60-word limit. GPT-4o-mini provided more technical detail, Gemini used beginner-friendly analogies, and Claude provided a balanced explanation.

This experiment demonstrates that models can satisfy one instruction while failing another. It also highlights the importance of validating generated responses against all specified constraints.

The prompt was suggested by ChatGPT and tested through all three models in Google Colab.

## **Final Submission Checklist**

- [x] Configured three LLM models from OpenAI, Google, and Anthropic through OpenRouter.
- [x] Successfully tested all three models.
- [x] Completed the Creative Writing task.
- [x] Completed the Analytical Reasoning task.
- [x] Completed the Instruction Following task.
- [x] Recorded actual model outputs and response times.
- [x] Implemented automatic formatting and validation checks.
- [x] Completed the comparative analysis of experimental results.
- [x] Completed manual comparison notes.
- [x] Added a detailed reflection based on experimental results.
- [x] Included AI usage disclosure.
- [x] Conducted an additional prompt comparison experiment.
- [x] Analyzed model strengths, limitations, and instruction-following failures.
- [x] Protected the API key using `getpass`.
- [x] Verified the reflection word count (700–900 words).
- [x] Uploaded the improved notebook to GitHub.